# Model-Based Reinforcement Learning

## Learning Objectives
- Estimate transition probabilities $\hat{P}_{sa}(s')$ and rewards $\hat{R}(s)$ from experience
- Understand MLE for $P_{sa}$: count ratio over observed transitions
- Implement the **integrated RL loop**: collect experience → estimate model → value iteration → update policy
- Understand the **exploration-exploitation** trade-off in model learning
- Compare model-based RL to direct policy search

## Problem Statement

In **model-based RL**, the MDP transitions $P_{sa}$ and reward $R$ are **unknown** and must be estimated from experience.

**MLE for transition probabilities.**  
Let $n_{sa}$ = number of times $(s, a)$ was visited, and $n_{sa}(s')$ = number of times $(s, a)$ led to $s'$:
$$\hat{P}_{sa}(s') = \frac{n_{sa}(s')}{n_{sa}}$$

If $n_{sa} = 0$, use a **uniform prior**: $\hat{P}_{sa}(s') = 1/|S|$.

**MLE for rewards.**  
$$\hat{R}(s) = \frac{\sum_t R_t \mathbf{1}[s_t = s]}{\sum_t \mathbf{1}[s_t = s]}$$

**Integrated model-based RL algorithm:**
1. Take action $a = \pi(s)$, observe $(s, a, r, s')$
2. Update counts: $n_{sa} \mathrel{+}= 1$, $n_{sa}(s') \mathrel{+}= 1$, reward accumulator
3. Recompute $\hat{P}_{sa}$ and $\hat{R}$ from counts
4. Run value iteration on $\hat{P}$, $\hat{R}$, $\gamma$ to get $\hat{V}^*$ and $\hat{\pi}^*$
5. Set $\pi \leftarrow \hat{\pi}^*$; go to 1

## 1. MLE Estimation of Transition Probabilities

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# True 4x4 grid world
grid_size  = 4
n_states   = grid_size ** 2
n_actions  = 4
goal_state = n_states - 1

def state_to_rc(s, g=grid_size): return s // g, s % g
def rc_to_state(r, c, g=grid_size): return r * g + c

def build_transitions(g=grid_size, slip=0.0):
    n = g * g
    P = np.zeros((n, 4, n))
    for s in range(n):
        r, c = state_to_rc(s, g)
        for a, (dr, dc) in enumerate([(-1,0),(1,0),(0,-1),(0,1)]):
            nr, nc = max(0, min(g-1, r+dr)), max(0, min(g-1, c+dc))
            ns = rc_to_state(nr, nc, g)
            P[s, a, ns] += (1.0 - slip)
            # With slip: random action
            for a2, (dr2, dc2) in enumerate([(-1,0),(1,0),(0,-1),(0,1)]):
                nr2, nc2 = max(0, min(g-1, r+dr2)), max(0, min(g-1, c+dc2))
                P[s, a, rc_to_state(nr2, nc2, g)] += slip / 4.0
    # Normalize
    P /= P.sum(axis=2, keepdims=True)
    return P

P_true = build_transitions(slip=0.15)  # unknown to agent
R_true = np.full(n_states, -1.0)
R_true[goal_state] = 10.0
gamma = 0.9

# Show how MLE estimate improves with more samples
np.random.seed(42)
n_sample_sizes = [10, 50, 200, 1000]
frob_errors = []

for n_samples in n_sample_sizes:
    n_sa    = np.zeros((n_states, n_actions))
    n_sas   = np.zeros((n_states, n_actions, n_states))

    # Collect random transitions
    for _ in range(n_samples):
        s = np.random.randint(n_states)
        a = np.random.randint(n_actions)
        s_next = np.random.choice(n_states, p=P_true[s, a])
        n_sa[s, a]      += 1
        n_sas[s, a, s_next] += 1

    # MLE estimate
    P_hat = np.where(n_sa[:, :, None] > 0,
                     n_sas / np.maximum(n_sa[:, :, None], 1),
                     1.0 / n_states)
    frob_errors.append(np.sqrt(((P_hat - P_true) ** 2).sum()))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Error vs sample size
ax = axes[0]
ax.loglog(n_sample_sizes, frob_errors, 'b-o', lw=2.5, ms=8)
# Reference 1/sqrt(n) curve
ref = [frob_errors[0] * np.sqrt(n_sample_sizes[0] / n) for n in n_sample_sizes]
ax.loglog(n_sample_sizes, ref, 'r--', lw=2, label='$O(1/\\sqrt{n})$ reference')
ax.set_xlabel('Number of samples')
ax.set_ylabel('Frobenius error $\\|\\hat{P} - P_{true}\\|_F$')
ax.set_title('MLE estimation accuracy vs sample size')
ax.legend(fontsize=9)

# True vs estimated P for one state-action pair
ax = axes[1]
s_show, a_show = 5, 3  # show state 5, action 3 (right)
ax.bar(range(n_states), P_true[s_show, a_show], color='steelblue', alpha=0.7,
       label='True $P_{sa}$')

# Use 200 samples
n_sa_200 = np.zeros((n_states, n_actions))
n_sas_200 = np.zeros((n_states, n_actions, n_states))
for _ in range(200):
    s_ = np.random.randint(n_states)
    a_ = np.random.randint(n_actions)
    s_n = np.random.choice(n_states, p=P_true[s_, a_])
    n_sa_200[s_, a_] += 1
    n_sas_200[s_, a_, s_n] += 1

P_hat_200 = np.where(n_sa_200[:, :, None] > 0,
                     n_sas_200 / np.maximum(n_sa_200[:, :, None], 1),
                     1.0 / n_states)
ax.bar(range(n_states), P_hat_200[s_show, a_show], color='#d6604d', alpha=0.5,
       label='Estimated $\\hat{P}_{sa}$ (n=200)')
ax.set_xlabel("Next state $s'$")
ax.set_ylabel('Probability')
ax.set_title(f'True vs estimated $P_{{sa}}$\nState {s_show}, action {a_show} (→)')
ax.legend(fontsize=9)

# Coverage: fraction of (s,a) pairs visited
ax = axes[2]
coverage = [(n_sa_200[:, :] > 0).mean() * 100]
ns = [10, 50, 100, 200, 500, 1000]
coverages = []
for n_s in ns:
    n_sa_tmp = np.zeros((n_states, n_actions))
    for _ in range(n_s):
        s_ = np.random.randint(n_states)
        a_ = np.random.randint(n_actions)
        n_sa_tmp[s_, a_] += 1
    coverages.append((n_sa_tmp > 0).mean() * 100)

ax.plot(ns, coverages, 'g-o', lw=2.5, ms=7)
ax.axhline(100, color='k', ls='--', lw=1.5, label='Full coverage')
ax.set_xlabel('Number of samples')
ax.set_ylabel('% of (s,a) pairs visited')
ax.set_title(f'Exploration coverage\n({n_states * n_actions} total (s,a) pairs)')
ax.legend(fontsize=9)

fig.suptitle('MLE Estimation of Transition Probabilities from Experience',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. The Integrated RL Algorithm

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def value_iteration(P, R, gamma, max_iters=500, tol=1e-6):
    V = np.zeros(len(R))
    for _ in range(max_iters):
        Q = R[:, None] + gamma * (P * V[None, None, :]).sum(axis=2)
        V_new = Q.max(axis=1)
        if np.max(np.abs(V_new - V)) < tol: break
        V = V_new
    return V, Q.argmax(axis=1)

def simulate_episode(s0, policy, P_env, R_env, max_steps=50, gamma=0.9):
    """Run one episode; return total discounted reward and trajectory."""
    s = s0
    total_r = 0.0
    trajectory = []
    for t in range(max_steps):
        a = policy[s]
        s_next = np.random.choice(n_states, p=P_env[s, a])
        r = R_env[s]
        total_r += (gamma ** t) * r
        trajectory.append((s, a, r, s_next))
        s = s_next
        if s == goal_state:
            break
    return total_r, trajectory

np.random.seed(7)

# Model estimates
n_sa   = np.zeros((n_states, n_actions))
n_sas  = np.zeros((n_states, n_actions, n_states))
r_sum  = np.zeros(n_states)
r_cnt  = np.zeros(n_states)

# Start with random policy
policy = np.random.randint(0, n_actions, n_states)
P_hat  = np.ones((n_states, n_actions, n_states)) / n_states
R_hat  = np.zeros(n_states)

n_update_freq = 5   # update model every N episodes
n_total_episodes = 200
episode_rewards = []
model_errors    = []

# Compute optimal policy on true MDP for reference
V_opt, pi_opt = value_iteration(P_true, R_true, gamma)

for ep in range(n_total_episodes):
    s0 = np.random.randint(n_states - 1)  # random start, not goal
    total_r, traj = simulate_episode(s0, policy, P_true, R_true, gamma=gamma)
    episode_rewards.append(total_r)

    # Update counts from trajectory
    for (s, a, r, s_next) in traj:
        n_sa[s, a]      += 1
        n_sas[s, a, s_next] += 1
        r_sum[s] += r
        r_cnt[s] += 1

    # Update model and policy periodically
    if (ep + 1) % n_update_freq == 0:
        P_hat = np.where(n_sa[:, :, None] > 0,
                         n_sas / np.maximum(n_sa[:, :, None], 1),
                         1.0 / n_states)
        R_hat = np.where(r_cnt > 0, r_sum / np.maximum(r_cnt, 1), 0.0)
        _, policy = value_iteration(P_hat, R_hat, gamma)
        model_errors.append(np.sqrt(((P_hat - P_true) ** 2).sum()))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Episode rewards over time (smoothed)
ax = axes[0]
window = 20
smoothed = np.convolve(episode_rewards, np.ones(window)/window, mode='valid')
ax.plot(episode_rewards, alpha=0.3, color='steelblue', lw=1, label='Raw')
ax.plot(range(window-1, n_total_episodes), smoothed, 'b-', lw=2.5, label=f'Smoothed (w={window})')
# Optimal policy reward for reference
opt_rewards = [simulate_episode(np.random.randint(n_states-1), pi_opt,
                                P_true, R_true, gamma=gamma)[0]
               for _ in range(50)]
ax.axhline(np.mean(opt_rewards), color='r', ls='--', lw=2, label='Optimal policy (avg)')
ax.set_xlabel('Episode')
ax.set_ylabel('Discounted return')
ax.set_title('Learning curve: model-based RL')
ax.legend(fontsize=8.5)

# Model error over time
ax = axes[1]
update_episodes = [(i+1) * n_update_freq for i in range(len(model_errors))]
ax.semilogy(update_episodes, model_errors, 'r-o', lw=2.5, ms=5)
ax.set_xlabel('Episode')
ax.set_ylabel('$\\|\\hat{P} - P_{true}\\|_F$ (log)')
ax.set_title('Model estimation error over time\n(decreases as more data collected)')

# Final estimated vs true value function
ax = axes[2]
V_hat, _ = value_iteration(P_hat, R_hat, gamma)
ax.scatter(V_opt, V_hat, s=40, alpha=0.8, c='steelblue')
mn, mx = min(V_opt.min(), V_hat.min()), max(V_opt.max(), V_hat.max())
ax.plot([mn, mx], [mn, mx], 'r--', lw=2, label='y=x (perfect)')
ax.set_xlabel('True $V^*(s)$')
ax.set_ylabel('Estimated $\\hat{V}^*(s)$')
err_v = np.max(np.abs(V_hat - V_opt))
ax.set_title(f'Estimated vs true $V^*$\n$\\|\\hat{{V}}^* - V^*\\|_\\infty = {err_v:.3f}$')
ax.legend(fontsize=9)

fig.suptitle(f'Integrated Model-Based RL ({n_total_episodes} episodes)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Exploration vs Exploitation

**Problem:** If we always follow $\hat{\pi}^*$, we may never visit states needed to improve $\hat{P}$.

**$\epsilon$-greedy exploration:** With probability $\epsilon$, take a random action instead of $\hat{\pi}^*(s)$.
This balances:
- **Exploitation:** acting on current best estimate
- **Exploration:** gathering new data to improve the model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def run_mbrl(P_env, R_env, gamma, n_episodes, epsilon, update_freq=5, seed=0):
    rng = np.random.default_rng(seed)
    n_s, n_a = n_states, n_actions
    n_sa  = np.zeros((n_s, n_a))
    n_sas = np.zeros((n_s, n_a, n_s))
    r_sum = np.zeros(n_s)
    r_cnt = np.zeros(n_s)
    policy = rng.integers(0, n_a, n_s)
    rewards = []

    for ep in range(n_episodes):
        s = rng.integers(0, n_s - 1)
        ep_r = 0.0
        for t in range(50):
            # epsilon-greedy
            if rng.random() < epsilon:
                a = rng.integers(0, n_a)
            else:
                a = policy[s]
            s_next = rng.choice(n_s, p=P_env[s, a])
            r = R_env[s]
            ep_r += (gamma ** t) * r
            n_sa[s, a] += 1
            n_sas[s, a, s_next] += 1
            r_sum[s] += r
            r_cnt[s] += 1
            s = s_next
            if s == goal_state: break
        rewards.append(ep_r)

        if (ep + 1) % update_freq == 0:
            P_hat = np.where(n_sa[:, :, None] > 0,
                             n_sas / np.maximum(n_sa[:, :, None], 1),
                             1.0 / n_s)
            R_hat = np.where(r_cnt > 0, r_sum / np.maximum(r_cnt, 1), 0.0)
            _, policy = value_iteration(P_hat, R_hat, gamma)
    return rewards

epsilons = [0.0, 0.05, 0.2, 0.5, 1.0]
n_ep = 300
window = 30

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(epsilons)))
final_perf = []
for eps, col in zip(epsilons, colors):
    rews = run_mbrl(P_true, R_true, gamma, n_ep, epsilon=eps, seed=42)
    sm   = np.convolve(rews, np.ones(window)/window, mode='valid')
    ax.plot(sm, color=col, lw=2.5, label=f'ε={eps}')
    final_perf.append(np.mean(rews[-50:]))

ax.axhline(np.mean(opt_rewards), color='r', ls='--', lw=2, label='Optimal')
ax.set_xlabel('Episode')
ax.set_ylabel('Smoothed discounted return')
ax.set_title(f'$\\epsilon$-greedy exploration (smoothed over {window} episodes)')
ax.legend(fontsize=8.5)

# Final performance vs epsilon
ax = axes[1]
ax.plot(epsilons, final_perf, 'b-o', lw=2.5, ms=8)
ax.axhline(np.mean(opt_rewards), color='r', ls='--', lw=2, label='Optimal policy')
ax.set_xlabel('Exploration probability $\\epsilon$')
ax.set_ylabel('Final performance (last 50 episodes)')
ax.set_title('Performance vs exploration rate\n(too little = poor model, too much = no exploitation)')
ax.legend(fontsize=9)

fig.suptitle('Exploration vs Exploitation in Model-Based RL',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Full Algorithm Visualisation: Policy Evolution

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

# Track policy at several checkpoints
checkpoints = [5, 15, 40, 100]
policy_snapshots = {}

n_sa_ck  = np.zeros((n_states, n_actions))
n_sas_ck = np.zeros((n_states, n_actions, n_states))
r_sum_ck = np.zeros(n_states)
r_cnt_ck = np.zeros(n_states)
pol_ck   = np.random.randint(0, n_actions, n_states)
epsilon  = 0.15

for ep in range(max(checkpoints) + 1):
    s = np.random.randint(n_states - 1)
    for t in range(50):
        a = np.random.randint(n_actions) if np.random.rand() < epsilon else pol_ck[s]
        s_next = np.random.choice(n_states, p=P_true[s, a])
        n_sa_ck[s, a] += 1
        n_sas_ck[s, a, s_next] += 1
        r_sum_ck[s] += R_true[s]
        r_cnt_ck[s] += 1
        s = s_next
        if s == goal_state: break

    if ep in checkpoints:
        P_ck = np.where(n_sa_ck[:, :, None] > 0,
                        n_sas_ck / np.maximum(n_sa_ck[:, :, None], 1),
                        1.0 / n_states)
        R_ck = np.where(r_cnt_ck > 0, r_sum_ck / np.maximum(r_cnt_ck, 1), 0.0)
        V_ck, pol_ck = value_iteration(P_ck, R_ck, gamma)
        policy_snapshots[ep] = (pol_ck.copy(), V_ck.copy())

action_arrows = ['↑', '↓', '←', '→']
action_deltas = [(-0.3, 0), (0.3, 0), (0, -0.3), (0, 0.3)]

fig, axes = plt.subplots(1, len(checkpoints) + 1, figsize=(4*(len(checkpoints)+1), 4.5))

for col, ep in enumerate(checkpoints):
    pol, V = policy_snapshots[ep]
    ax = axes[col]
    im = ax.imshow(V.reshape(grid_size, grid_size), cmap='YlGn', alpha=0.6)
    for s in range(n_states):
        if s == goal_state:
            r, c = state_to_rc(s)
            ax.scatter(c, r, s=300, c='gold', marker='*', zorder=5)
            continue
        r, c = state_to_rc(s)
        a = pol[s]
        dr, dc = action_deltas[a]
        ax.annotate('', xy=(c+dc, r+dr), xytext=(c, r),
                    arrowprops=dict(arrowstyle='->', color='#2166ac', lw=1.8,
                                   mutation_scale=12))
    match = np.mean(pol == pi_opt) * 100
    ax.set_title(f'After {ep} episodes\n{match:.0f}% match optimal', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

# Optimal policy
ax = axes[len(checkpoints)]
im = ax.imshow(V_opt.reshape(grid_size, grid_size), cmap='YlGn', alpha=0.6)
for s in range(n_states):
    if s == goal_state:
        r, c = state_to_rc(s)
        ax.scatter(c, r, s=300, c='gold', marker='*', zorder=5)
        continue
    r, c = state_to_rc(s)
    a = pi_opt[s]
    dr, dc = action_deltas[a]
    ax.annotate('', xy=(c+dc, r+dr), xytext=(c, r),
                arrowprops=dict(arrowstyle='->', color='red', lw=1.8, mutation_scale=12))
ax.set_title('True optimal π*\n(ground truth)', fontsize=9)
ax.set_xticks([]); ax.set_yticks([])

fig.suptitle('Policy Evolution During Model-Based RL (ε=0.15)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Derivation Pathway

### Derivation pathway

In [ ]:
from IPython.display import HTML
HTML("""
<svg xmlns="http://www.w3.org/2000/svg" width="780" height="468"
     viewBox="0 0 780 468" font-family="Segoe UI, Arial, sans-serif">
  <defs>
    <marker id="ah" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#444"/>
    </marker>
    <marker id="ahd" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#999"/>
    </marker>
  </defs>
  <rect x="10" y="12" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="35" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Interact with</text>
  <text x="102" y="52" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">environment</text>
  <line x1="197" y1="35" x2="216" y2="35"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="12" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="40" font-size="13" text-anchor="middle" fill="#333"
        >observe (s, a, s&#x2019;) transitions; unknown P&#x209b;&#x2090; and R(s)</text>
  <line x1="102" y1="58" x2="102" y2="120"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="80" font-size="11.5" font-style="italic" fill="#555"
        >estimate model from transitions</text>
  <rect x="10" y="112" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="135" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">MLE model</text>
  <text x="102" y="152" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">P&#x209b;&#x2090;, R&#x209c;&#x1d4f;</text>
  <line x1="197" y1="135" x2="216" y2="135"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="112" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="140" font-size="13" text-anchor="middle" fill="#333"
        >P&#x209b;&#x2090;(s&#x2019;) = count(s,a,s&#x2019;)/count(s,a); R&#x209c;&#x1d4f;(s) = avg observed reward</text>
  <line x1="102" y1="158" x2="102" y2="220"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="180" font-size="11.5" font-style="italic" fill="#555"
        >run value iteration on estimated model</text>
  <rect x="10" y="212" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="235" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">VI on model</text>
  <text x="102" y="252" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">&#x2192; &#x3c0;&#x207b;&#x1d57;&#x207e;</text>
  <line x1="197" y1="235" x2="216" y2="235"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="212" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="240" font-size="13" text-anchor="middle" fill="#333"
        >compute V* under current model estimate; extract greedy policy</text>
  <line x1="102" y1="258" x2="102" y2="320"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="280" font-size="11.5" font-style="italic" fill="#555"
        >execute &#x3c0;&#x207b;&#x1d57;&#x207e;, collect new data</text>
  <rect x="10" y="312" width="185" height="46" rx="7"
        fill="#1a5fa8" stroke="#1a5fa8" stroke-width="2"/>
  <text x="102" y="335" font-size="12.5" font-weight="700"
        text-anchor="middle" fill="#ffffff">Converge</text>
  <text x="102" y="352" font-size="12.5" font-weight="700"
        text-anchor="middle" fill="#ffffff">to optimal &#x3c0;*</text>
  <line x1="197" y1="335" x2="216" y2="335"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="312" width="548" height="46" rx="7"
        fill="#dce8f8" stroke="#7aadd4" stroke-width="1.5"/>
  <text x="495" y="340" font-size="13" text-anchor="middle" fill="#333"
        >as data grows P&#x209b;&#x2090; and R&#x209c;&#x1d4f; &#x2192; true values; &#x3c0;&#x207b;&#x1d57;&#x207e; &#x2192; &#x3c0;*</text>
</svg>
""")

## Summary

| Concept | Formula / Description | Key Insight |
|---|---|---|
| Transition MLE | $\hat{P}_{sa}(s') = \frac{\#(s,a,s')}{\#(s,a)}$ | Count-based estimate; converges as data grows |
| Reward estimate | $\hat{R}(s) = $ average reward observed in state $s$ | Simple sample mean |
| Model-based loop | Interact → estimate model → value iteration → act | Alternates exploration and planning |
| Exploration | Random actions ensure all $(s,a)$ visited | Necessary for MLE estimates to converge |
| vs. model-free | Learns explicit $P_{sa}$, then solves exactly | Sample-efficient but requires model storage |

**Key insight:** model-based RL separates learning (fitting the MDP from data) from planning (solving the MDP with value iteration); as the model estimate converges to the true transition probabilities, the resulting policy converges to the optimal one.